# Eddy Kinectic Energy

This code aims to analyze how Eddy kinectic energy changes with resolution. We will try to analyze that through SSH variations, and through EKE calcualtions itself. It is importnat to highigh however, that sd(SSH) can also exhibit a signal when there is DSW outflow ( Matthias work). Sp SSH is not necessairily a clear comparisson

In [1]:
import cosima_cookbook as cc
from cosima_cookbook import distributed as ccd
import matplotlib.pyplot as plt
import numpy as np
import netCDF4 as nc
import xarray as xr
import glob,os
import cmocean.cm as cmocean

import logging
logging.captureWarnings(True)
logging.getLogger('py.warnings').setLevel(logging.ERROR)

from dask.distributed import Client

In [2]:
client = Client(n_workers=32)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 32
Total threads: 32,Total memory: 2.95 TiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34575,Workers: 32
Dashboard: http://127.0.0.1:8787/status,Total threads: 32
Started: Just now,Total memory: 2.95 TiB
Comm: tcp://127.0.0.1:37881,Total threads: 1
Dashboard: http://127.0.0.1:42751/status,Memory: 94.49 GiB
Nanny: tcp://127.0.0.1:44169,


Session and experiments settings

In [3]:
exp = 'panant-01-zstar-ACCESSyr2'
start_time= '2000-01-01'
end_time= '2000-12-31'
lat_range = slice(-90,-59)
isobath_depth = 1000
rho_0 = 1035.0

session = cc.database.create_session()


Calculating u' and v', vertically integrated for panan01

In [9]:
exp05 = 'panant-005-zstar-ACCESSyr2'
depth_slice=slice(200,1000)#  calculating just for the upper 1000 m
#Panan01, U and V
u005 = cc.querying.getvar(exp05, 'uo', session,start_time=start_time, end_time=end_time,frequency='1 daily').sel(time=slice(start_time,end_time)).sel(yh=lat_range).sel(z_l_sub01=depth_slice)

v005 = cc.querying.getvar(exp05, 'vo', session,start_time=start_time, end_time=end_time,frequency='1 daily').sel(time=slice(start_time,end_time)).sel(yq=lat_range).sel(z_l_sub01=depth_slice)

#Volcello for vertical average
Vol005 = cc.querying.getvar(exp05, 'volcello', session,start_time=start_time, end_time=end_time,ncfile='%daily_z%').sel(time=slice(start_time,end_time)).sel(yh=lat_range).sel(z_l=depth_slice)

Vol005_u = Vol005.interp(xh=u005.xq).rename({'z_l':'z_l_sub01'})
Vol005_v = Vol005.interp(yh=v005.yq).rename({'z_l':'z_l_sub01'})

#Averaging
u005m=u005.weighted(Vol005_u.fillna(0)).mean('z_l_sub01')
v005m=v005.weighted(Vol005_v.fillna(0)).mean('z_l_sub01')

#calcualting u' and v'
u005p=u005m - u005m.mean('time')
v005p=v005m - v005m.mean('time')

# #loading data
u005p = u005p.compute()
v005p = v005p.compute()

In [ ]:
u005pc=u005p.interp(xq=Vol005.xh)
v005pc=v005p.interp(yq=Vol005.yh)
del u005p,v005p
EKE005=0.5*((u005pc**2) + (v005pc**2))
EKE005m=EKE005.mean('time')

Now for panan01

In [ ]:
#Panan01, U and V
u01 = cc.querying.getvar(exp, 'uo', session,start_time=start_time, end_time=end_time,frequency='1 daily').sel(time=slice(start_time,end_time)).sel(yh=lat_range).sel(z_l_sub01=depth_slice)

v01 = cc.querying.getvar(exp, 'vo', session,start_time=start_time, end_time=end_time,frequency='1 daily').sel(time=slice(start_time,end_time)).sel(yq=lat_range).sel(z_l_sub01=depth_slice)

#Volcello for vertical average
Vol01 = cc.querying.getvar(exp, 'volcello', session,start_time=start_time, end_time=end_time,ncfile='%daily_z%').sel(time=slice(start_time,end_time)).sel(yh=lat_range).sel(z_l=depth_slice)

Vol01_u = Vol01.interp(xh=u01.xq).rename({'z_l':'z_l_sub01'})
Vol01_v = Vol01.interp(yh=v01.yq).rename({'z_l':'z_l_sub01'})

#Averaging
u01m=u01.weighted(Vol01_u.fillna(0)).mean('z_l_sub01')
v01m=v01.weighted(Vol01_v.fillna(0)).mean('z_l_sub01')

#calcualting u' and v'
u01p=u01m - u01m.mean('time')
v01p=v01m - v01m.mean('time')

#loading data
u01p = u01p.compute()
v01p = v01p.compute()

interpolating into the centre grid, and calculating EKE01

In [ ]:
u01pc=u01p.interp(xq=Vol01.xh)
v01pc=v01p.interp(yq=Vol01.yh)
del u01p,v01p
EKE01=0.5*((u01pc**2) + (v01pc**2))
EKE01m=EKE01.mean('time')

Calculating mean EKE

In [ ]:
mEKE005 = EKE005.mean('time')
mEKE01 = EKE01.mean('time')

In [ ]:
#Interping 01 into 005 for calculating differences
mEKE01_on005=mEKE01.interp(xh=mEKE005.xh,yh=mEKE005.yh)

In [ ]:
#importing depth for masking and locating the isobath - panan01
ht01 = cc.querying.getvar(exp, 'thetao', session,n=1).isel(time=0)
ht01 = ht01.z_l + (ht01*0)
ht01 = ht01.max('z_l')
ht01.load()
ht01_0=ht01.fillna(0)
ht01_mask=ht01.where(ht01<2000)*0
ht01_mask2500=ht01.where(ht01>2500)*0
ht01_land=ht01_0.where(ht01_0<=0)

ht005 = cc.querying.getvar(exp05, 'thetao', session,n=1).isel(time=0)
ht005 = ht005.z_l + (ht005*0)
ht005 = ht005.max('z_l')
ht005.load()
ht005_0=ht005.fillna(0)
ht005_mask=ht005.where(ht005<2000)*0
ht005_mask2500=ht005.where(ht005>2500)*0
ht005_land=ht005_0.where(ht005_0<=0)

## Mean EKE in different resolutions

Calculate for the uppper 1000m

In [ ]:
import cmocean as cm
import cartopy.crs as ccrs
import cosima_cookbook as cc
import cartopy.feature as cft

land_10m = cft.NaturalEarthFeature('physical', 'land', '10m',
                                   edgecolor='black', facecolor='papayawhip', linewidth=0.5)

In [ ]:
figpath='/home/156/wf4500/v45_wf4500/Project_panan/GH/Panan_HT_ASC/figs/'

### Ross Sea

In [ ]:
RSlons=[-200,-150]
plt.figure(figsize=(10, 12))
plt.subplots_adjust(left=0.1,
                    bottom=0.1, 
                    right=0.9, 
                    top=0.9, 
                    wspace=0.4, 
                    hspace=0.4)

plt.subplot(3,1,1)
(mEKE01).plot(x='xh', y='yh',
         vmin=0, vmax=0.01, extend='both',
         cmap='jet',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 15,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)


plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(RSlons); plt.ylim(-80,-68)
plt.title(r"[a] Panan-$0.1^o$")

plt.subplot(3,1,2)
(mEKE005).plot(x='xh', y='yh',
         vmin=0, vmax=0.01, extend='both',
         cmap='jet',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 15,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)
plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(RSlons); plt.ylim(-80,-68)
plt.title(r"[b] Panan-$0.05^o$")


plt.subplot(3,1,3)
(mEKE005 - mEKE01_on005).plot(x='xh', y='yh',
         vmin=-0.01, vmax=0.01, extend='both',
         cmap='seismic',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 15,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)
plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(RSlons); plt.ylim(-80,-68)
plt.title(r"[c] Panan-$0.05^o$ - Panan-$0.1^o$")

plt.savefig(figpath+'EKE_ROSS.png',dpi=300)


In the **Ross sea** we see a very marked and interesting increase in EKE along the 1km isobath. That is very suggestive of an increase in Eddy flow across the isobath in higher resolution. This agrees with the increase **2 TW** increase in Southward Eddy heat transport in the Ross Sea ( check CSHT_resolutions_totaltimescales.ipynb)

### East Antarctica

In [ ]:
plt.figure(figsize=(10, 12))
plt.subplots_adjust(left=0.1,
                    bottom=0.1, 
                    right=0.9, 
                    top=0.9, 
                    wspace=0.4, 
                    hspace=0.4)

plt.subplot(3,1,1)
(mEKE01).plot(x='xh', y='yh',
         vmin=0, vmax=0.01, extend='both',
         cmap='jet',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 15,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)


plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(-280,-200); plt.ylim(-70,-64)
plt.title(r"[a] Panan-$0.1^o$")

plt.subplot(3,1,2)
(mEKE005).plot(x='xh', y='yh',
         vmin=0, vmax=0.01, extend='both',
         cmap='jet',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 15,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)
plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(-280,-200); plt.ylim(-70,-64)
plt.title(r"[b] Panan-$0.05^o$")


plt.subplot(3,1,3)
(mEKE005 - mEKE01_on005).plot(x='xh', y='yh',
         vmin=-0.01, vmax=0.01, extend='both',
         cmap='seismic',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 15,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)
plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(-280,-200); plt.ylim(-70,-64)
plt.title(r"[c] Panan-$0.05^o$ - Panan-$0.1^o$")


plt.savefig(figpath+'EKE_EA.png',dpi=300)

In **East Antarctica**, t seems to me that on the higher resolution, the centres of eddy activity changed. FOr example, the coastal current seem to have higher EKE in most of the embayments. There is also a session between -230 and -250 where we have incrased EKE along the 1km isobath. This doesnt give us a very good idea on why the Eddy heat transport changes direction in the model. Maybe locating the ASC in the two resolutions might help

### Weddell Sea

In [ ]:
WSlons=[-65,-20]
plt.figure(figsize=(12, 6))
plt.subplots_adjust(left=0.1,
                    bottom=0.1, 
                    right=0.9, 
                    top=0.9, 
                    wspace=0.6, 
                    hspace=0.4)

plt.subplot(1,3,1)
(mEKE01).plot(x='xh', y='yh',
         vmin=0, vmax=0.005, extend='both',
         cmap='jet',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 30,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)


plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(WSlons); plt.ylim(-80,-62)
plt.title(r"[a] Panan-$0.1^o$")

plt.subplot(1,3,2)
(mEKE005).plot(x='xh', y='yh',
         vmin=0, vmax=0.005, extend='both',
         cmap='jet',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 30,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)
plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(WSlons); plt.ylim(-80,-62)
plt.title(r"[b] Panan-$0.05^o$")


plt.subplot(1,3,3)
(mEKE005 - mEKE01_on005).plot(x='xh', y='yh',
         vmin=-0.003, vmax=0.003, extend='both',
         cmap='seismic',
         cbar_kwargs = {'label': r"EKE [$m^2/s^2$]",
                        'fraction': 0.03,
                        'aspect': 30,
                        'shrink': 0.7});
plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)
plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(WSlons); plt.ylim(-80,-62)
plt.title(r"[c] Panan-$0.05^o$ - Panan-$0.1^o$")

plt.savefig(figpath+'EKE_Weddell.png',dpi=300)


In the **Weddell sea** it seems that the signal is less evident too. In general, when looking at a-b, it seems that there is an increase in EKE along the 1km isobath. The difference plot shows some dislocation of the EKE centres too, which likely hass to doo with changes in ASC position. 

ps: We have to be carefull with the difference maps tho, because for that we had to onterpolate panan01, into panan005 grid.

Summary: it seems that along the 1km isobath, there are increase in the EKE along the Ross Sea, and Weddell Sea, and these are locations where we observed increase SOuthward Eddy heat transort when calculating regional budgets. This help us confirm that the Eddy heat transport chages observes are due to changes in Eddy flow. 

**Can we draw further interpreataions from these graphs?**

## ASC Speed and lcoation with resolution

Here the objective it is to check how the ASC llcates in relation to the isobath in both resolutions

In [ ]:
# #calcualting u and v in the upper 500 m
# depth_slice2=slice(0,500)
# u005_up=u005.sel(z_l_sub01=depth_slice2)
# v005_up=v005.sel(z_l_sub01=depth_slice2)
# Vol005_uup = Vol005.interp(xh=u005.xq).sel(z_l=depth_slice2).rename({'z_l':'z_l_sub01'})
# Vol005_vup = Vol005.interp(yh=v005.yq).sel(z_l=depth_slice2).rename({'z_l':'z_l_sub01'})



In [ ]:
#calculating mean speed, conserving the direction of u
u005signal=(u005pc.where(u005pc<0)*0)-1; u005signal=u005signal.fillna(1)
speed005 = (( (v005pc**2) + (u005pc**2) )**0.5) * (u005signal)
u01signal=(u01pc.where(u01pc<0)*0)-1; u01signal=u01signal.fillna(1)
speed01 = (( (v01pc**2) + (u01pc**2) )**0.5) * (u01signal)

#time means
mspeed005=speed005.groupby('time.month').mean('time').mean('month')
mspeed01=speed01.groupby('time.month').mean('time').mean('month')
#Interping 01 into 005 for calculating differences
mspeed01_on005=mspeed01.interp(xh=mspeed005.xh,yh=mspeed005.yh)

In [ ]:
plt.figure(figsize=(10, 12))
plt.subplots_adjust(left=0.1,
                    bottom=0.1, 
                    right=0.9, 
                    top=0.9, 
                    wspace=0.4, 
                    hspace=0.4)

plt.subplot(3,1,1)
(mspeed01).plot(vmin=-0.02,vmax=0.02, extend='both',
         cmap='seismic',
         cbar_kwargs = {'label': r"Speed [$m/s$]",
                        'fraction': 0.03,
                        'aspect': 15,
                        'shrink': 0.7})

plt.contourf(ht01_land.xh,ht01_land.yh,ht01_land,colors='grey',linewidh=0.5,alpha=0.7)


plt.contour(ht01_0.xh,ht01_0.yh,ht01_0,[1000],colors='grey',linewidh=1)
plt.xlim(-280,-200); plt.ylim(-70,-64)
plt.title(r"[a] Panan-$0.1^o$")
